# MS-CGCA Ablation: What Is Actually Producing the Gain?

The causal MS-CGCA three-way ensemble at a 60-beat window reached macro-F1 0.6825 — essentially
matching the non-causal reference (0.682) while remaining deployable. That is a strong result, but
**three things changed simultaneously** relative to the previous causal run (F1 0.596):

| Change | Previously known effect |
|---|---|
| Window 120 → 60 beats | ConfigB work measured +0.043 for this alone |
| CNN-BiLSTM-Attention → MS-CGCA | untested in isolation |
| Two-way → three-way ensemble | rejected twice before, for non-reproduction and for losing to its own best component |

With all three changed at once, "MS-CGCA improves performance" is not supported by the evidence. The
60-beat window is a known-good contributor and may account for much of the gain on its own.

---

## What this notebook answers

```
A. Does 0.6825 reproduce under a different seed?
B. How much comes from the window change alone?
C. How much comes from the MS-CGCA architecture?
D. Is the three-way ensemble actually needed, or does
   two-way suffice?
```

Six configurations, each isolating one factor:

| # | Architecture | Window | Ensemble | Isolates |
|---|---|---|---|---|
| 1 | MS-CGCA | 60 | three-way | reproduction of the headline result |
| 2 | MS-CGCA | 60 | two-way | value of the third (fine-tuned) voter |
| 3 | MS-CGCA | 120 | two-way | effect of window size |
| 4 | BiLSTM (old) | 60 | two-way | effect of architecture |
| 5 | BiLSTM (old) | 120 | two-way | the previous causal baseline (≈0.596) |
| 6 | XGBoost only | 60 | — | how much the network contributes at all |

Configurations 2 and 4 differ only in architecture; 2 and 3 differ only in window. Together they
separate the two factors that were previously confounded.

---

## Honest note on the three-way configuration

Configurations using the fine-tuned voter evaluate on windows remaining after each subject's
calibration slice is removed (approximately 9,510 rather than the full set). This is
**subject-adaptive evaluation, not strict leave-one-subject-out** — the third voter trains on 20% of
the held-out subject's own labelled data. Two-way configurations use the full evaluation set and are
strict LOSO. The two are therefore not directly comparable, and the table reports the evaluation
window count for each so this is visible rather than implied.


## 1. Setup

In [1]:
!pip install neurokit2 xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 36.2 MB/s eta 0:00:00


In [2]:
import os, pickle, warnings, json, time
import numpy as np, pandas as pd
from scipy.signal import welch
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, cohen_kappa_score
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk
warnings.filterwarnings('ignore')

# ---- SEED: change this to test reproducibility ----
SEED = 7                    # the headline run used 42
np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_PATH='/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE='/kaggle/working'; os.makedirs(SAVE,exist_ok=True)
SUBJECT_IDS=[2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES=['relaxed','mild','moderate','high']; NCLS=4

STEP=5
EWMA_HALFLIVES={'fast':60,'medium':300,'slow':1800}
POPULATION_RR_MS=780.0
ROLL_WINDOW=20
NOISE_BAND=0.03

RESULTS=f'{SAVE}/ablation_seed{SEED}.json'
print("TF",tf.__version__,"GPU",len(tf.config.list_physical_devices('GPU'))>0)
print("SEED",SEED)

TF 2.20.0 GPU True
SEED 7


## 2. Pipeline — copied verbatim from the MS-CGCA notebook

Preprocessing, causal replacements and feature functions are reproduced exactly. Any difference here
would make the ablation measure something other than what it claims to.

In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f:
        data=pickle.load(f,encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg=nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _,info=nk.ecg_peaks(ecg, sampling_rate=fs); rp=info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr, ts):
    rr=rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)]=np.nan
    for i in range(1,len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1]>0.20: rr[i]=np.nan
    m=np.isnan(rr)
    if m.any(): rr[m]=np.interp(np.where(m)[0],np.where(~m)[0],rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap=np.interp(rp/fe, np.arange(len(wt))/ft, wt); return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out=[]
    for i in range(len(rp)-1):
        seg=labels[rp[i]:rp[i+1]]; v=seg[seg>0]
        out.append(0 if len(v)==0 else np.bincount(v).argmax())
    return np.array(out)

def ewma_causal(x, halflife):
    a=1-np.exp(np.log(0.5)/max(halflife,1))
    o=np.empty(len(x),dtype=float); state=float(POPULATION_RR_MS)
    for i in range(len(x)):
        state=a*x[i]+(1-a)*state; o[i]=state
    return o

def causal_zscore(x, halflife=300):
    a=1-np.exp(np.log(0.5)/max(halflife,1))
    mu=np.empty(len(x)); sd=np.empty(len(x))
    m=float(x[0]) if len(x) else 0.0; v=1.0
    for i in range(len(x)):
        d=x[i]-m; m=m+a*d; v=(1-a)*(v+a*d*d)
        mu[i]=m; sd[i]=np.sqrt(max(v,1e-8))
    return (x-mu)/(sd+1e-8)

def roll_rmssd_causal(x, w=ROLL_WINDOW):
    o=np.zeros(len(x))
    for i in range(len(x)):
        seg=x[max(0,i-w+1):i+1]
        o[i]=np.sqrt(np.mean(np.diff(seg)**2)) if len(seg)>1 else 0.0
    return o

def roll_sdnn_causal(x, w=ROLL_WINDOW):
    o=np.zeros(len(x))
    for i in range(len(x)):
        seg=x[max(0,i-w+1):i+1]
        o[i]=np.std(seg) if len(seg)>1 else 0.0
    return o

def hrv_features(w, fs=4.0):
    rr,diff=np.array(w),np.diff(w)
    mean_rr=np.mean(rr); sdnn=np.std(rr); rmssd=np.sqrt(np.mean(diff**2))
    pnn50=np.sum(np.abs(diff)>50)/len(diff)*100; cv=sdnn/mean_rr
    t=np.cumsum(rr)/1000.0; u=np.interp(np.arange(0,t[-1],1/fs),t,rr)
    fr,psd=welch(u,fs=fs,nperseg=min(256,len(u)))
    vlf=TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf=TRAPZ(psd[(fr>=0.04)&(fr<0.15)])
    hf=TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf=lf/(hf+1e-8); lf_nu=lf/(lf+hf+1e-8)
    sd1=np.sqrt(0.5)*np.std(diff); sd2=np.sqrt(max(2*sdnn**2-0.5*np.var(diff),0))
    return np.array([mean_rr,sdnn,rmssd,pnn50,cv,vlf,lf,hf,lf_hf,lf_nu,sd1,sd2,sd1/(sd2+1e-8)])

def resid_features(rw):
    r=np.array(rw)
    return np.array([np.mean(r),np.std(r),np.max(np.abs(r)),
                     np.polyfit(np.arange(len(r)),r,1)[0], np.sum(r**2)/len(r)])

def circ_features(ts):
    t,hour=ts%86400,(ts%86400)/3600.0
    cort=0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),cort])

def circ7(ts):
    t=ts%86400; hour=t/3600.0
    return np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
        np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),
        0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2),
        np.sin(2*np.pi*(hour-23)/24),np.cos(2*np.pi*(hour-23)/24)])
print("pipeline defined")

pipeline defined


### Load subjects once, build feature sets per window size

In [4]:
raw={}
for sid in SUBJECT_IDS:
    try:
        chest,wt,labels=load_subject(sid); ecg=chest['ECG'].flatten()
        rr,ts,rp=extract_rr_from_ecg(ecg); temp=align_temp(wt,rp); rr,ts=clean_rr(rr,ts)
        rl=labels_to_rr(labels,rp); keep=rl>0
        rrk,tk,tsk,lk=rr[keep],temp[keep],ts[keep],rl[keep]
        new=np.zeros(len(lk),dtype=int); si=np.where(lk==2)[0]
        if len(si)>0:
            srr=rrk[si]; loc=[]
            for i in range(len(srr)):
                w=srr[max(0,i-15):i+15]; dd=np.diff(w)
                loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc=np.array(loc); p33,p66=np.percentile(loc,33),np.percentile(loc,66)
            for i,idx in enumerate(si): new[idx]=(1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        raw[sid]=dict(rr=rrk,temp=tk,ts=tsk,lab=new)
    except Exception as e: print("FAIL",sid,e)
print(len(raw),"subjects loaded"); assert len(raw)==15

def build(window):
    Xs,Xc,Xx,y,g=[],[],[],[],[]
    for sid,d in raw.items():
        rr,temp,ts,labels=d['rr'],d['temp'],d['ts'],d['lab']
        base={k:ewma_causal(rr,hl) for k,hl in EWMA_HALFLIVES.items()}
        res_med=rr-base['medium']
        temp_res=temp-ewma_causal(temp,EWMA_HALFLIVES['medium'])
        rn=causal_zscore(rr); rm=roll_rmssd_causal(rn); sd=roll_sdnn_causal(rn)
        hr=60000/(rr+1e-8); rrn=causal_zscore(res_med)
        tn=causal_zscore(temp); trn=causal_zscore(temp_res)
        for s in range(0,len(rr)-window,STEP):
            e=s+window; mid=s+window//2; bi=min(mid,len(ts)-1)
            try:
                xf=np.concatenate([hrv_features(rr[s:e]),resid_features(res_med[s:e]),
                                   np.array([base['fast'][e-1],base['slow'][e-1]]),
                                   circ_features(ts[bi])])
            except Exception:
                continue
            Xs.append(np.stack([rn[s:e],rm[s:e],sd[s:e],hr[s:e],rrn[s:e],tn[s:e],trn[s:e]],axis=-1))
            Xc.append(circ7(ts[bi])); Xx.append(xf)
            y.append(labels[mid]); g.append(sid)
    return (np.array(Xs,np.float32),np.array(Xc,np.float32),np.array(Xx),
            np.array(y,np.int32),np.array(g,np.int32))

DATA={}
for w in [60,120]:
    t0=time.time()
    DATA[w]=build(w)
    print(f"window {w:3d}: seq {DATA[w][0].shape}  xgb {DATA[w][2].shape}  "
          f"classes {np.bincount(DATA[w][3])}  ({time.time()-t0:.0f}s)")

15 subjects loaded
window  60: seq (12026, 60, 7)  xgb (12026, 25)  classes [8774 1101 1080 1071]  (12s)
window 120: seq (11846, 120, 7)  xgb (11846, 25)  classes [8594 1101 1080 1071]  (14s)


## 3. Both architectures

MS-CGCA is copied verbatim from the new notebook. The BiLSTM model is the previous causal
architecture, included so the two can be compared at the same window size.

In [5]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma=gamma
    def call(self,yt,yp):
        yt=tf.cast(yt,tf.int32)
        ce=tf.keras.losses.sparse_categorical_crossentropy(yt,yp)
        pt=tf.reduce_sum(tf.one_hot(yt,4)*yp,axis=-1)
        return tf.pow(1.0-pt,self.gamma)*ce

def build_ms_cgca(window, nch=7, ncirc=7, ncls=4):
    """Verbatim from notebook-newmodel. Causal by construction:
    padding='causal' convolutions and a unidirectional LSTM.
    Note: attention shapes align because MaxPooling1D(2) and
    RepeatVector(window//2) both halve — adding a second pooling
    layer would silently desync query and key."""
    si=layers.Input(shape=(window,nch),name='sequence')
    ci=layers.Input(shape=(ncirc,),name='circadian')
    c1=layers.Conv1D(32,3,padding='causal',activation='relu',dilation_rate=1)(si)
    c2=layers.Conv1D(32,3,padding='causal',activation='relu',dilation_rate=2)(si)
    c4=layers.Conv1D(32,3,padding='causal',activation='relu',dilation_rate=4)(si)
    x=layers.Concatenate()([c1,c2,c4])
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.LSTM(128,return_sequences=True)(x)
    x=layers.Dropout(0.4)(x)
    cp=layers.Dense(128,activation='relu')(ci)
    cp=layers.RepeatVector(window//2)(cp)
    attn=layers.Attention()([cp,x])
    x=layers.GlobalAveragePooling1D()(attn)
    cf=layers.Dense(32,activation='relu')(ci)
    m=layers.Concatenate()([x,cf])
    o=layers.Dense(64,activation='relu')(m); o=layers.Dropout(0.4)(o)
    o=layers.Dense(ncls,activation='softmax')(o)
    return Model([si,ci],o,name='MS_CGCA')

def build_bilstm(window, nch=7, ncirc=7, ncls=4):
    """The previous architecture. NOTE: the bidirectional LSTM reads
    backwards through time, so this model is not causal by design even
    when fed causal features. Included only for architectural comparison."""
    si=layers.Input(shape=(window,nch),name='sequence')
    ci=layers.Input(shape=(ncirc,),name='circadian')
    x=layers.Conv1D(64,7,padding='same',activation='relu')(si)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(0.4)(x); a=layers.Attention()([x,x])
    x=layers.GlobalAveragePooling1D()(a)
    c=layers.Dense(32,activation='relu')(ci); c=layers.Dense(16,activation='relu')(c)
    x=layers.Concatenate()([x,c]); x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    return Model([si,ci],layers.Dense(ncls,activation='softmax')(x),name='BiLSTM')

def make_xgb():
    return XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.8,objective='multi:softprob',num_class=4,
        eval_metric='mlogloss',random_state=SEED,n_jobs=-1)

def freeze_extractor(m):
    for l in m.layers:
        ln=l.__class__.__name__.lower()
        l.trainable = not any(k in ln for k in ['conv','lstm','batchnorm','attention','pooling'])
    return m

def strat_calib(sub_idx, y, frac=0.2, minpc=5):
    lab=y[sub_idx]; ntot=int(len(sub_idx)*frac); calib=[]
    rng=np.random.RandomState(42)
    for c in np.unique(lab):
        pos=sub_idx[lab==c]; take=min(max(minpc,ntot//len(np.unique(lab))),len(pos))
        calib.extend(rng.choice(pos,take,replace=False))
    calib=np.array(sorted(calib)); ev=np.array([i for i in sub_idx if i not in set(calib)])
    return calib,ev
print("architectures defined")

architectures defined


## 4. The ablation runner

One function handles all six configurations. Two-way runs evaluate on the full held-out subject;
three-way runs exclude the calibration slice and report the reduced window count.

In [6]:
def macro_f1(yt,yp,K=4):
    f=[]
    for c in range(K):
        tp=np.sum((yp==c)&(yt==c)); fp=np.sum((yp==c)&(yt!=c)); fn=np.sum((yp!=c)&(yt==c))
        f.append(0.0 if tp==0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(f))

def run_config(arch, window, three_way, label, xgb_only=False):
    Xs,Xc,Xx,y,g=DATA[window]
    logo=LeaveOneGroupOut(); folds=[]
    for tr,te in logo.split(Xx,y,g):
        if three_way:
            calib,ev=strat_calib(te,y)
        else:
            calib,ev=None,te

        sc=StandardScaler().fit(Xx[tr])
        xgb=make_xgb()
        xgb.fit(sc.transform(Xx[tr]),y[tr],
                sample_weight=compute_sample_weight('balanced',y[tr]),verbose=False)
        p_xgb=xgb.predict_proba(sc.transform(Xx[ev]))

        if xgb_only:
            folds.append(dict(y=y[ev],p_xgb=p_xgb,p_cnn=None,p_ft=None)); continue

        cw=compute_class_weight('balanced',classes=np.unique(y[tr]),y=y[tr])
        net=(build_ms_cgca if arch=='mscgca' else build_bilstm)(window)
        net.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=SparseFocalLoss(2.0))
        cb=[callbacks.EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True,verbose=0),
            callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,verbose=0)]
        net.fit([Xs[tr],Xc[tr]],y[tr],validation_split=0.15,epochs=120,batch_size=32,
                class_weight=dict(enumerate(cw)),callbacks=cb,verbose=0)
        p_cnn=net.predict([Xs[ev],Xc[ev]],verbose=0)

        p_ft=None
        if three_way:
            ft=(build_ms_cgca if arch=='mscgca' else build_bilstm)(window)
            ft.set_weights(net.get_weights()); ft=freeze_extractor(ft)
            ft.compile(optimizer=tf.keras.optimizers.Adam(5e-5),loss=SparseFocalLoss(2.0))
            if len(np.unique(y[calib]))>=2:
                cwc=compute_class_weight('balanced',classes=np.unique(y[calib]),y=y[calib])
                cwdc=dict(zip(np.unique(y[calib]),cwc))
            else: cwdc=None
            ft.fit([Xs[calib],Xc[calib]],y[calib],epochs=15,batch_size=8,
                   class_weight=cwdc,verbose=0)
            p_ft=ft.predict([Xs[ev],Xc[ev]],verbose=0)

        folds.append(dict(y=y[ev],p_xgb=p_xgb,p_cnn=p_cnn,p_ft=p_ft))
        tf.keras.backend.clear_session()
        print('.',end='',flush=True)

    # nested weight selection (never uses the outer subject)
    if xgb_only:
        yt=np.concatenate([f['y'] for f in folds])
        yp=np.concatenate([f['p_xgb'].argmax(1) for f in folds])
    else:
        if three_way:
            GRID=[(wf,round(wx*(1-wf),4),round((1-wx)*(1-wf),4))
                  for wf in [0.0,0.1,0.2,0.3] for wx in np.arange(0.3,0.71,0.1)]
            def blend(f,w):
                wf,wx,wc=w
                return wx*f['p_xgb']+wc*f['p_cnn']+wf*f['p_ft']
        else:
            GRID=[(0.0,round(wx,4),round(1-wx,4)) for wx in np.arange(0.3,0.71,0.05)]
            def blend(f,w):
                _,wx,wc=w
                return wx*f['p_xgb']+wc*f['p_cnn']
        preds,ys=[],[]
        for i,fS in enumerate(folds):
            inner=[f for j,f in enumerate(folds) if j!=i]
            innerY=np.concatenate([f['y'] for f in inner])
            best=max(GRID,key=lambda w: macro_f1(innerY,
                     np.concatenate([blend(f,w).argmax(1) for f in inner])))
            preds.append(blend(fS,best).argmax(1)); ys.append(fS['y'])
        yt=np.concatenate(ys); yp=np.concatenate(preds)

    res=dict(label=label,arch=arch,window=window,three_way=three_way,
             n_eval=int(len(yt)),
             f1=macro_f1(yt,yp),
             kappa=float(cohen_kappa_score(yt,yp,weights='quadratic')),
             acc=float(np.mean(yt==yp)))
    print(f"  {label:<34} F1={res['f1']:.4f} kappa={res['kappa']:.4f} n={res['n_eval']}")
    return res
print("runner ready")

runner ready


## 5. Run all six configurations

In [7]:
CONFIGS=[
 ('mscgca',  60, True,  'MS-CGCA  60  three-way', False),
 ('mscgca',  60, False, 'MS-CGCA  60  two-way',   False),
 ('mscgca', 120, False, 'MS-CGCA 120  two-way',   False),
 ('bilstm',  60, False, 'BiLSTM   60  two-way',   False),
 ('bilstm', 120, False, 'BiLSTM  120  two-way',   False),
 ('none',    60, False, 'XGBoost  60  alone',     True),
]

store=json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
t0=time.time()
for arch,w,tw,label,xo in CONFIGS:
    if label in store:
        print(f"{label:<34} cached F1={store[label]['f1']:.4f}"); continue
    print(f"\n{label}"); 
    store[label]=run_config(arch,w,tw,label,xgb_only=xo)
    json.dump(store,open(RESULTS,'w'))
print(f"\n\ntotal elapsed {(time.time()-t0)/60:.1f} min")


MS-CGCA  60  three-way


I0000 00:00:1786510070.211298      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786510070.214883      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


...............  MS-CGCA  60  three-way             F1=0.6832 kappa=0.8241 n=9650

MS-CGCA  60  two-way
...............  MS-CGCA  60  two-way               F1=0.6523 kappa=0.8020 n=12026

MS-CGCA 120  two-way
...............  MS-CGCA 120  two-way               F1=0.6184 kappa=0.8169 n=11846

BiLSTM   60  two-way
...............  BiLSTM   60  two-way               F1=0.6626 kappa=0.7990 n=12026

BiLSTM  120  two-way
...............  BiLSTM  120  two-way               F1=0.5875 kappa=0.7717 n=11846

XGBoost  60  alone
  XGBoost  60  alone                 F1=0.6549 kappa=0.8072 n=12026


total elapsed 104.3 min


## 6. Results

In [8]:
rows=[store[c[3]] for c in CONFIGS if c[3] in store]
df=pd.DataFrame(rows)[['label','window','three_way','n_eval','f1','kappa','acc']]
print("="*82)
print(f"ABLATION RESULTS  (seed {SEED})")
print("="*82)
print(f"{'configuration':<26}{'win':>5}{'3way':>6}{'n_eval':>8}{'F1':>9}{'kappa':>9}{'acc':>8}")
print("-"*82)
for _,r in df.iterrows():
    print(f"{r.label:<26}{r.window:>5}{str(r.three_way):>6}{r.n_eval:>8}"
          f"{r.f1:>9.4f}{r.kappa:>9.4f}{r.acc:>8.4f}")
print("="*82)
print("\nREFERENCE:")
print(f"  headline MS-CGCA 60 three-way (seed 42)   F1=0.6825  kappa=0.8497")
print(f"  previous causal BiLSTM 120 two-way        F1=0.596   kappa=0.786")
print(f"  non-causal BiLSTM 120 two-way (offline)   F1=0.682   kappa=0.855")

ABLATION RESULTS  (seed 7)
configuration               win  3way  n_eval       F1    kappa     acc
----------------------------------------------------------------------------------
MS-CGCA  60  three-way       60  True    9650   0.6832   0.8241  0.9171
MS-CGCA  60  two-way         60 False   12026   0.6523   0.8020  0.8527
MS-CGCA 120  two-way        120 False   11846   0.6184   0.8169  0.8440
BiLSTM   60  two-way         60 False   12026   0.6626   0.7990  0.8563
BiLSTM  120  two-way        120 False   11846   0.5875   0.7717  0.8304
XGBoost  60  alone           60 False   12026   0.6549   0.8072  0.8539

REFERENCE:
  headline MS-CGCA 60 three-way (seed 42)   F1=0.6825  kappa=0.8497
  previous causal BiLSTM 120 two-way        F1=0.596   kappa=0.786
  non-causal BiLSTM 120 two-way (offline)   F1=0.682   kappa=0.855


In [9]:
def get(label,key='f1'):
    return store[label][key] if label in store else None

print("="*70); print("ATTRIBUTION"); print("="*70)

a=get('MS-CGCA  60  three-way'); b=get('MS-CGCA  60  two-way')
c=get('MS-CGCA 120  two-way');   d=get('BiLSTM   60  two-way')
e=get('BiLSTM  120  two-way');   x=get('XGBoost  60  alone')

if a is not None:
    print(f"\nA. REPRODUCTION (seed {SEED} vs seed 42)")
    print(f"   this run  {a:.4f}   headline 0.6825   diff {a-0.6825:+.4f}")
    print(f"   {'reproduces within noise' if abs(a-0.6825)<NOISE_BAND else 'DOES NOT reproduce - differs beyond the noise band'}")

if b is not None and c is not None:
    print(f"\nB. WINDOW SIZE (MS-CGCA two-way, 60 vs 120)")
    print(f"   60 beats  {b:.4f}    120 beats {c:.4f}    effect {b-c:+.4f}")

if b is not None and d is not None:
    print(f"\nC. ARCHITECTURE (two-way at 60 beats)")
    print(f"   MS-CGCA   {b:.4f}    BiLSTM    {d:.4f}    effect {b-d:+.4f}")

if a is not None and b is not None:
    print(f"\nD. THIRD VOTER (MS-CGCA at 60 beats)")
    print(f"   three-way {a:.4f}    two-way   {b:.4f}    effect {a-b:+.4f}")
    print(f"   NOTE: n_eval differs ({store['MS-CGCA  60  three-way']['n_eval']} vs "
          f"{store['MS-CGCA  60  two-way']['n_eval']}) — three-way is")
    print(f"   subject-adaptive, not strict LOSO. Not a like-for-like comparison.")

if x is not None and b is not None:
    print(f"\nE. NETWORK CONTRIBUTION (60 beats)")
    print(f"   XGBoost alone {x:.4f}   + MS-CGCA {b:.4f}   effect {b-x:+.4f}")

print(f"\nAll effects should be read against the +/-{NOISE_BAND} reseeding band.")

ATTRIBUTION

A. REPRODUCTION (seed 7 vs seed 42)
   this run  0.6832   headline 0.6825   diff +0.0007
   reproduces within noise

B. WINDOW SIZE (MS-CGCA two-way, 60 vs 120)
   60 beats  0.6523    120 beats 0.6184    effect +0.0339

C. ARCHITECTURE (two-way at 60 beats)
   MS-CGCA   0.6523    BiLSTM    0.6626    effect -0.0103

D. THIRD VOTER (MS-CGCA at 60 beats)
   three-way 0.6832    two-way   0.6523    effect +0.0309
   NOTE: n_eval differs (9650 vs 12026) — three-way is
   subject-adaptive, not strict LOSO. Not a like-for-like comparison.

E. NETWORK CONTRIBUTION (60 beats)
   XGBoost alone 0.6549   + MS-CGCA 0.6523   effect -0.0026

All effects should be read against the +/-0.03 reseeding band.


## 7. Interpreting this

**If the headline reproduces (A within ±0.03)**, the 0.6825 figure is stable and can be reported.
If it does not, a single run was not sufficient evidence and the configuration needs more seeds
before any claim is made about it.

**Window versus architecture (B and C).** The ConfigB work independently measured +0.043 for
shortening the window from 120 to 60 beats with the old architecture. If B recovers a similar
figure and C is small, the gain is primarily the window change and MS-CGCA should not be credited
with it. If C is substantial, the architecture is contributing genuinely — and given that MS-CGCA
replaces a bidirectional LSTM (which reads backwards through time and was never deployable) with a
causal unidirectional one, an architectural gain here would be a meaningful result rather than an
incidental one.

**The third voter (D).** Read this with the evaluation counts in view. The three-way configuration
is evaluated on fewer windows because each subject's calibration slice is withheld, and the third
voter has trained on that subject's own labelled data. A positive effect here is not evidence that
the ensemble generalises better — it partly reflects having seen the test subject. This
configuration was rejected twice previously for related reasons, and a small positive effect is not
sufficient to reverse that.

**Network contribution (E).** If XGBoost alone is close to the two-way result, the network is adding
little and the simpler deployment is preferable — fewer dependencies, no TensorFlow at inference,
and one model to version rather than two.

**What none of this changes.** The mechanism findings are unaffected. This is a deployment question:
which configuration ships, and what its honest performance is.
